# 🏪🛒🏷️⬇️STORE SALES FORECASTING💰💲📊📈💸

This is a continuation notebook from the analysis cleaning and feature engineering of the store sales dataset. This notebook focuses solely on modelling an forecasting or predictions using a `HybridModel`(LinearRegression + XgboostRegressor. The `linearRegression` model would be made to learn the trend and the `XgboostRegressor` model would be made to learn the seasons, cycles and residual components of our timeseries.

## IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from statsmodels.graphics.tsaplots import plot_pacf, seasonal_plot
from scipy.signal import periodogram
from itertools import product

In [2]:
# Display all the columns in the dataset
pd.pandas.set_option('display.max_columns', None)

# Display all the rows in the dataset
# pd.pandas.set_option('display.max_rows', None)

## LOAD DATA

In [3]:
# load the train set
train_df = pd.read_csv("train_set.csv",
                       low_memory = False,
                       dtype = {
                           'store_nbr' : 'category',
                           'family' : 'category',
                           'sales' : 'float32',
                           'onpromotion' : 'uint32',
                           'city' : 'category',
                           'state' : 'category',
                           'type' : 'category',
                           'cluster' : 'category'
                       },
                       parse_dates = ['date'],
                      )

train_df['date'] = train_df.date.dt.to_period('D')
train_df = train_df.set_index(['store_nbr', 'family', 'date']).sort_index()
train_df

id      sales  onpromotion  \
store_nbr family     date                                          
1         AUTOMOTIVE 2013-01-01        0   0.000000            0   
                     2013-01-02     1782   2.000000            0   
                     2013-01-03     3564   3.000000            0   
                     2013-01-04     5346   3.000000            0   
                     2013-01-05     7128   5.000000            0   
...                                  ...        ...          ...   
9         SEAFOOD    2017-08-11  2993759  23.830999            0   
                     2017-08-12  2995541  16.859001            4   
                     2017-08-13  2997323  20.000000            0   
                     2017-08-14  2999105  17.000000            0   
                     2017-08-15  3000887  16.000000            0   

                                 is_regional_holiday  is_local_holiday   city  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2013-01-01                False             False  Quito   
                     2013-01-02                False             False  Quito   
                     2013-01-03                False             False  Quito   
                     2013-01-04                False             False  Quito   
                     2013-01-05                False             False  Quito   
...                                              ...               ...    ...   
9         SEAFOOD    2017-08-11                False             False  Quito   
                     2017-08-12                False             False  Quito   
                     2017-08-13                False             False  Quito   
                     2017-08-14                False             False  Quito   
                     2017-08-15                False             False  Quito   

                                     state type cluster  const   trend  \
store_nbr family     date                                                
1         AUTOMOTIVE 2013-01-01  Pichincha    D      13    1.0     1.0   
                     2013-01-02  Pichincha    D      13    1.0     2.0   
                     2013-01-03  Pichincha    D      13    1.0     3.0   
                     2013-01-04  Pichincha    D      13    1.0     4.0   
                     2013-01-05  Pichincha    D      13    1.0     5.0   
...                                    ...  ...     ...    ...     ...   
9         SEAFOOD    2017-08-11  Pichincha    B       6    1.0  1684.0   
                     2017-08-12  Pichincha    B       6    1.0  1685.0   
                     2017-08-13  Pichincha    B       6    1.0  1686.0   
                     2017-08-14  Pichincha    B       6    1.0  1687.0   
                     2017-08-15  Pichincha    B       6    1.0  1688.0   

                                 s(2,7)  s(3,7)  s(4,7)  s(5,7)  s(6,7)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2013-01-01     0.0     0.0     0.0     0.0     0.0   
                     2013-01-02     1.0     0.0     0.0     0.0     0.0   
                     2013-01-03     0.0     1.0     0.0     0.0     0.0   
                     2013-01-04     0.0     0.0     1.0     0.0     0.0   
                     2013-01-05     0.0     0.0     0.0     1.0     0.0   
...                                 ...     ...     ...     ...     ...   
9         SEAFOOD    2017-08-11     0.0     0.0     1.0     0.0     0.0   
                     2017-08-12     0.0     0.0     0.0     1.0     0.0   
                     2017-08-13     0.0     0.0     0.0     0.0     1.0   
                     2017-08-14     0.0     0.0     0.0     0.0     0.0   
                     2017-08-15     0.0     0.0     0.0     0.0     0.0   

                                 s(7,7)  sin(1,freq=YE-DEC)  \
store_nbr family     date                                     
1         AUTOMOTIVE 2013-01-01   

In [4]:
# Load the test set
test_df = pd.read_csv("test_set.csv",
                      low_memory = False,
                      dtype = {
                          'store_nbr' : 'category',
                          'family' : 'category',
                          'onpromotion' : 'uint32',
                          'city' : 'category',
                          'state' : 'category',
                          'type' : 'category',
                          'cluster' : 'category'
                      },
                      parse_dates = ['date'],
                     )

test_df['date'] = test_df.date.dt.to_period('D')
test_df = test_df.set_index(['store_nbr', 'family', 'date']).sort_index()
test_df

id  onpromotion  is_local_holiday  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16  3000888            0             False   
                     2017-08-17  3002670            0             False   
                     2017-08-18  3004452            0             False   
                     2017-08-19  3006234            0             False   
                     2017-08-20  3008016            0             False   
...                                  ...          ...               ...   
9         SEAFOOD    2017-08-27  3022271            0             False   
                     2017-08-28  3024053            0             False   
                     2017-08-29  3025835            0             False   
                     2017-08-30  3027617            0             False   
                     2017-08-31  3029399            0             False   

                                  city      state type cluster  const   trend  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2017-08-16  Quito  Pichincha    D      13    1.0  1689.0   
                     2017-08-17  Quito  Pichincha    D      13    1.0  1690.0   
                     2017-08-18  Quito  Pichincha    D      13    1.0  1691.0   
                     2017-08-19  Quito  Pichincha    D      13    1.0  1692.0   
                     2017-08-20  Quito  Pichincha    D      13    1.0  1693.0   
...                                ...        ...  ...     ...    ...     ...   
9         SEAFOOD    2017-08-27  Quito  Pichincha    B       6    1.0  1700.0   
                     2017-08-28  Quito  Pichincha    B       6    1.0  1701.0   
                     2017-08-29  Quito  Pichincha    B       6    1.0  1702.0   
                     2017-08-30  Quito  Pichincha    B       6    1.0  1703.0   
                     2017-08-31  Quito  Pichincha    B       6    1.0  1704.0   

                                 s(2,7)  s(3,7)  s(4,7)  s(5,7)  s(6,7)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16     1.0     0.0     0.0     0.0     0.0   
                     2017-08-17     0.0     1.0     0.0     0.0     0.0   
                     2017-08-18     0.0     0.0     1.0     0.0     0.0   
                     2017-08-19     0.0     0.0     0.0     1.0     0.0   
                     2017-08-20     0.0     0.0     0.0     0.0     1.0   
...                                 ...     ...     ...     ...     ...   
9         SEAFOOD    2017-08-27     0.0     0.0     0.0     0.0     1.0   
                     2017-08-28     0.0     0.0     0.0     0.0     0.0   
                     2017-08-29     0.0     0.0     0.0     0.0     0.0   
                     2017-08-30     1.0     0.0     0.0     0.0     0.0   
                     2017-08-31     0.0     1.0     0.0     0.0     0.0   

                                 s(7,7)  sin(1,freq=YE-DEC)  \
store_nbr family     date                                     
1         AUTOMOTIVE 2017-08-16     0.0           -0.693281   
                     2017-08-17     0.0           -0.705584   
                     2017-08-18     0.0           -0.717677   
                     2017-08-19     0.0           -0.729558   
                     2017-08-20     0.0           -0.741222   
...                                 ...                 ...   
9         SEAFOOD    2017-08-27     0.0           -0.816538   
                     2017-08-28     1.0           -0.826354   
                     2017-08-29     0.0           -0.835925   
                     2017-08-30     0.0           -0.845249   
                     2017-08-31     0.0           -0.854322   

                                 cos(1,freq=YE-DEC)  sin(2,freq=YE-DEC)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16           -0.720667           

In [5]:
# Convert all bool types to int
train_df[train_df.select_dtypes(include = 'bool').columns] = train_df.select_dtypes(include = 'bool').astype(int)
train_df

id      sales  onpromotion  \
store_nbr family     date                                          
1         AUTOMOTIVE 2013-01-01        0   0.000000            0   
                     2013-01-02     1782   2.000000            0   
                     2013-01-03     3564   3.000000            0   
                     2013-01-04     5346   3.000000            0   
                     2013-01-05     7128   5.000000            0   
...                                  ...        ...          ...   
9         SEAFOOD    2017-08-11  2993759  23.830999            0   
                     2017-08-12  2995541  16.859001            4   
                     2017-08-13  2997323  20.000000            0   
                     2017-08-14  2999105  17.000000            0   
                     2017-08-15  3000887  16.000000            0   

                                 is_regional_holiday  is_local_holiday   city  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2013-01-01                    0                 0  Quito   
                     2013-01-02                    0                 0  Quito   
                     2013-01-03                    0                 0  Quito   
                     2013-01-04                    0                 0  Quito   
                     2013-01-05                    0                 0  Quito   
...                                              ...               ...    ...   
9         SEAFOOD    2017-08-11                    0                 0  Quito   
                     2017-08-12                    0                 0  Quito   
                     2017-08-13                    0                 0  Quito   
                     2017-08-14                    0                 0  Quito   
                     2017-08-15                    0                 0  Quito   

                                     state type cluster  const   trend  \
store_nbr family     date                                                
1         AUTOMOTIVE 2013-01-01  Pichincha    D      13    1.0     1.0   
                     2013-01-02  Pichincha    D      13    1.0     2.0   
                     2013-01-03  Pichincha    D      13    1.0     3.0   
                     2013-01-04  Pichincha    D      13    1.0     4.0   
                     2013-01-05  Pichincha    D      13    1.0     5.0   
...                                    ...  ...     ...    ...     ...   
9         SEAFOOD    2017-08-11  Pichincha    B       6    1.0  1684.0   
                     2017-08-12  Pichincha    B       6    1.0  1685.0   
                     2017-08-13  Pichincha    B       6    1.0  1686.0   
                     2017-08-14  Pichincha    B       6    1.0  1687.0   
                     2017-08-15  Pichincha    B       6    1.0  1688.0   

                                 s(2,7)  s(3,7)  s(4,7)  s(5,7)  s(6,7)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2013-01-01     0.0     0.0     0.0     0.0     0.0   
                     2013-01-02     1.0     0.0     0.0     0.0     0.0   
                     2013-01-03     0.0     1.0     0.0     0.0     0.0   
                     2013-01-04     0.0     0.0     1.0     0.0     0.0   
                     2013-01-05     0.0     0.0     0.0     1.0     0.0   
...                                 ...     ...     ...     ...     ...   
9         SEAFOOD    2017-08-11     0.0     0.0     1.0     0.0     0.0   
                     2017-08-12     0.0     0.0     0.0     1.0     0.0   
                     2017-08-13     0.0     0.0     0.0     0.0     1.0   
                     2017-08-14     0.0     0.0     0.0     0.0     0.0   
                     2017-08-15     0.0     0.0     0.0     0.0     0.0   

                                 s(7,7)  sin(1,freq=YE-DEC)  \
store_nbr family     date                                     
1         AUTOMOTIVE 2013-01-01   

In [6]:
# Convert all bool types to int
test_df[test_df.select_dtypes(include = 'bool').columns] = test_df.select_dtypes(include = 'bool').astype(int)
test_df

id  onpromotion  is_local_holiday  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16  3000888            0                 0   
                     2017-08-17  3002670            0                 0   
                     2017-08-18  3004452            0                 0   
                     2017-08-19  3006234            0                 0   
                     2017-08-20  3008016            0                 0   
...                                  ...          ...               ...   
9         SEAFOOD    2017-08-27  3022271            0                 0   
                     2017-08-28  3024053            0                 0   
                     2017-08-29  3025835            0                 0   
                     2017-08-30  3027617            0                 0   
                     2017-08-31  3029399            0                 0   

                                  city      state type cluster  const   trend  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2017-08-16  Quito  Pichincha    D      13    1.0  1689.0   
                     2017-08-17  Quito  Pichincha    D      13    1.0  1690.0   
                     2017-08-18  Quito  Pichincha    D      13    1.0  1691.0   
                     2017-08-19  Quito  Pichincha    D      13    1.0  1692.0   
                     2017-08-20  Quito  Pichincha    D      13    1.0  1693.0   
...                                ...        ...  ...     ...    ...     ...   
9         SEAFOOD    2017-08-27  Quito  Pichincha    B       6    1.0  1700.0   
                     2017-08-28  Quito  Pichincha    B       6    1.0  1701.0   
                     2017-08-29  Quito  Pichincha    B       6    1.0  1702.0   
                     2017-08-30  Quito  Pichincha    B       6    1.0  1703.0   
                     2017-08-31  Quito  Pichincha    B       6    1.0  1704.0   

                                 s(2,7)  s(3,7)  s(4,7)  s(5,7)  s(6,7)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16     1.0     0.0     0.0     0.0     0.0   
                     2017-08-17     0.0     1.0     0.0     0.0     0.0   
                     2017-08-18     0.0     0.0     1.0     0.0     0.0   
                     2017-08-19     0.0     0.0     0.0     1.0     0.0   
                     2017-08-20     0.0     0.0     0.0     0.0     1.0   
...                                 ...     ...     ...     ...     ...   
9         SEAFOOD    2017-08-27     0.0     0.0     0.0     0.0     1.0   
                     2017-08-28     0.0     0.0     0.0     0.0     0.0   
                     2017-08-29     0.0     0.0     0.0     0.0     0.0   
                     2017-08-30     1.0     0.0     0.0     0.0     0.0   
                     2017-08-31     0.0     1.0     0.0     0.0     0.0   

                                 s(7,7)  sin(1,freq=YE-DEC)  \
store_nbr family     date                                     
1         AUTOMOTIVE 2017-08-16     0.0           -0.693281   
                     2017-08-17     0.0           -0.705584   
                     2017-08-18     0.0           -0.717677   
                     2017-08-19     0.0           -0.729558   
                     2017-08-20     0.0           -0.741222   
...                                 ...                 ...   
9         SEAFOOD    2017-08-27     0.0           -0.816538   
                     2017-08-28     1.0           -0.826354   
                     2017-08-29     0.0           -0.835925   
                     2017-08-30     0.0           -0.845249   
                     2017-08-31     0.0           -0.854322   

                                 cos(1,freq=YE-DEC)  sin(2,freq=YE-DEC)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16           -0.720667           

In [7]:
# Get a list of the columns and rearrange it
col_list = list(test_df.columns)
col_list

['id',
 'onpromotion',
 'is_local_holiday',
 'city',
 'state',
 'type',
 'cluster',
 'const',
 'trend',
 's(2,7)',
 's(3,7)',
 's(4,7)',
 's(5,7)',
 's(6,7)',
 's(7,7)',
 'sin(1,freq=YE-DEC)',
 'cos(1,freq=YE-DEC)',
 'sin(2,freq=YE-DEC)',
 'cos(2,freq=YE-DEC)',
 'sin(3,freq=YE-DEC)',
 'cos(3,freq=YE-DEC)',
 'sin(4,freq=YE-DEC)',
 'cos(4,freq=YE-DEC)',
 'sin(5,freq=YE-DEC)',
 'cos(5,freq=YE-DEC)',
 'sin(1,freq=QE-DEC)',
 'cos(1,freq=QE-DEC)',
 'sin(2,freq=QE-DEC)',
 'cos(2,freq=QE-DEC)',
 'is_new_year',
 'dcoilwtico',
 'is_national_holiday',
 'is_regional_holiday',
 'holiday_type_Additional',
 'holiday_type_Bridge',
 'holiday_type_Event',
 'holiday_type_Holiday',
 'holiday_type_Transfer',
 'holiday_type_Work Day',
 'holiday_transferred',
 'is_holiday',
 'is_christmas',
 'is_payday_15',
 'is_payday_end',
 'is_payday',
 'earthquake_impact',
 'earthquake_decay',
 'is_new_year_lead(1)',
 'is_new_year_lead(2)',
 'is_new_year_lead(3)',
 'is_new_year_lag(1)',
 'is_new_year_lag(2)',
 'is_christ

In [8]:
# List for test set
test_list = [
     'city',
     'state',
     'type',
     'cluster',
     'const',
     'trend',
     'id',
     'onpromotion',
     's(2,7)',
     's(3,7)',
     's(4,7)',
     's(5,7)',
     's(6,7)',
     's(7,7)',
     'sin(1,freq=YE-DEC)',
     'cos(1,freq=YE-DEC)',
     'sin(2,freq=YE-DEC)',
     'cos(2,freq=YE-DEC)',
     'sin(3,freq=YE-DEC)',
     'cos(3,freq=YE-DEC)',
     'sin(4,freq=YE-DEC)',
     'cos(4,freq=YE-DEC)',
     'sin(5,freq=YE-DEC)',
     'cos(5,freq=YE-DEC)',
     'sin(1,freq=QE-DEC)',
     'cos(1,freq=QE-DEC)',
     'sin(2,freq=QE-DEC)',
     'cos(2,freq=QE-DEC)',
     'dcoilwtico',
     'dcoilwtico_lag',
     'is_national_holiday',
     'is_regional_holiday',
     'is_local_holiday',
     'holiday_type_Additional',
     'holiday_type_Bridge',
     'holiday_type_Event',
     'holiday_type_Holiday',
     'holiday_type_Transfer',
     'holiday_type_Work Day',
     'holiday_transferred',
     'is_holiday_lead',
     'is_holiday',
     'is_holiday_lag',
     'is_christmas_lead(2)',
     'is_christmas_lead(1)',
     'is_christmas',
     'is_christmas_lag(1)',
     'is_christmas_lag(2)',
     'is_christmas_lag(3)',
     'is_new_year_lead(3)',
     'is_new_year_lead(2)',
     'is_new_year_lead(1)',
     'is_new_year',
     'is_new_year_lag(1)',
     'is_new_year_lag(2)',
     'is_payday_15',
     'is_payday_end',
     'is_payday_lead',
     'is_payday',
     'is_payday_lag',
     'earthquake_impact',
     'earthquake_decay',
]

In [9]:
test_df = test_df[test_list]
test_df

city      state type cluster  const   trend  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2017-08-16  Quito  Pichincha    D      13    1.0  1689.0   
                     2017-08-17  Quito  Pichincha    D      13    1.0  1690.0   
                     2017-08-18  Quito  Pichincha    D      13    1.0  1691.0   
                     2017-08-19  Quito  Pichincha    D      13    1.0  1692.0   
                     2017-08-20  Quito  Pichincha    D      13    1.0  1693.0   
...                                ...        ...  ...     ...    ...     ...   
9         SEAFOOD    2017-08-27  Quito  Pichincha    B       6    1.0  1700.0   
                     2017-08-28  Quito  Pichincha    B       6    1.0  1701.0   
                     2017-08-29  Quito  Pichincha    B       6    1.0  1702.0   
                     2017-08-30  Quito  Pichincha    B       6    1.0  1703.0   
                     2017-08-31  Quito  Pichincha    B       6    1.0  1704.0   

                                      id  onpromotion  s(2,7)  s(3,7)  s(4,7)  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2017-08-16  3000888            0     1.0     0.0     0.0   
                     2017-08-17  3002670            0     0.0     1.0     0.0   
                     2017-08-18  3004452            0     0.0     0.0     1.0   
                     2017-08-19  3006234            0     0.0     0.0     0.0   
                     2017-08-20  3008016            0     0.0     0.0     0.0   
...                                  ...          ...     ...     ...     ...   
9         SEAFOOD    2017-08-27  3022271            0     0.0     0.0     0.0   
                     2017-08-28  3024053            0     0.0     0.0     0.0   
                     2017-08-29  3025835            0     0.0     0.0     0.0   
                     2017-08-30  3027617            0     1.0     0.0     0.0   
                     2017-08-31  3029399            0     0.0     1.0     0.0   

                                 s(5,7)  s(6,7)  s(7,7)  sin(1,freq=YE-DEC)  \
store_nbr family     date                                                     
1         AUTOMOTIVE 2017-08-16     0.0     0.0     0.0           -0.693281   
                     2017-08-17     0.0     0.0     0.0           -0.705584   
                     2017-08-18     0.0     0.0     0.0           -0.717677   
                     2017-08-19     1.0     0.0     0.0           -0.729558   
                     2017-08-20     0.0     1.0     0.0           -0.741222   
...                                 ...     ...     ...                 ...   
9         SEAFOOD    2017-08-27     0.0     1.0     0.0           -0.816538   
                     2017-08-28     0.0     0.0     1.0           -0.826354   
                     2017-08-29     0.0     0.0     0.0           -0.835925   
                     2017-08-30     0.0     0.0     0.0           -0.845249   
                     2017-08-31     0.0     0.0     0.0           -0.854322   

                                 cos(1,freq=YE-DEC)  sin(2,freq=YE-DEC)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2017-08-16           -0.720667            0.999250   
                     2017-08-17           -0.708627            0.999991   
                     2017-08-18           -0.696376            0.999546   
                     2017-08-19           -0.683919            0.997917   
                     2017-08-20           -0.671260            0.995105   
...                                             ...                 ...   
9         SEAFOOD    2017-08-27           -0.577292            0.942761   
                     2017-08-28           -0.563151            0.930724   
                     2017-08-29           -0.548843            0.917584   
                     2017-08-30           -0.534373            0.903356  

In [10]:
# List for train set
train_list = [
     'city',
     'state',
     'type',
     'cluster',
     'const',
     'trend',
     'id',
     'onpromotion',
     's(2,7)',
     's(3,7)',
     's(4,7)',
     's(5,7)',
     's(6,7)',
     's(7,7)',
     'sin(1,freq=YE-DEC)',
     'cos(1,freq=YE-DEC)',
     'sin(2,freq=YE-DEC)',
     'cos(2,freq=YE-DEC)',
     'sin(3,freq=YE-DEC)',
     'cos(3,freq=YE-DEC)',
     'sin(4,freq=YE-DEC)',
     'cos(4,freq=YE-DEC)',
     'sin(5,freq=YE-DEC)',
     'cos(5,freq=YE-DEC)',
     'sin(1,freq=QE-DEC)',
     'cos(1,freq=QE-DEC)',
     'sin(2,freq=QE-DEC)',
     'cos(2,freq=QE-DEC)',
     'dcoilwtico',
     'dcoilwtico_lag',
     'is_national_holiday',
     'is_regional_holiday',
     'is_local_holiday',
     'holiday_type_Additional',
     'holiday_type_Bridge',
     'holiday_type_Event',
     'holiday_type_Holiday',
     'holiday_type_Transfer',
     'holiday_type_Work Day',
     'holiday_transferred',
     'is_holiday_lead',
     'is_holiday',
     'is_holiday_lag',
     'is_christmas_lead(2)',
     'is_christmas_lead(1)',
     'is_christmas',
     'is_christmas_lag(1)',
     'is_christmas_lag(2)',
     'is_christmas_lag(3)',
     'is_new_year_lead(3)',
     'is_new_year_lead(2)',
     'is_new_year_lead(1)',
     'is_new_year',
     'is_new_year_lag(1)',
     'is_new_year_lag(2)',
     'is_payday_15',
     'is_payday_end',
     'is_payday_lead',
     'is_payday',
     'is_payday_lag',
     'earthquake_impact',
     'earthquake_decay',
     'sales'
]

In [11]:
train_df = train_df[train_list]
train_df

city      state type cluster  const   trend  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2013-01-01  Quito  Pichincha    D      13    1.0     1.0   
                     2013-01-02  Quito  Pichincha    D      13    1.0     2.0   
                     2013-01-03  Quito  Pichincha    D      13    1.0     3.0   
                     2013-01-04  Quito  Pichincha    D      13    1.0     4.0   
                     2013-01-05  Quito  Pichincha    D      13    1.0     5.0   
...                                ...        ...  ...     ...    ...     ...   
9         SEAFOOD    2017-08-11  Quito  Pichincha    B       6    1.0  1684.0   
                     2017-08-12  Quito  Pichincha    B       6    1.0  1685.0   
                     2017-08-13  Quito  Pichincha    B       6    1.0  1686.0   
                     2017-08-14  Quito  Pichincha    B       6    1.0  1687.0   
                     2017-08-15  Quito  Pichincha    B       6    1.0  1688.0   

                                      id  onpromotion  s(2,7)  s(3,7)  s(4,7)  \
store_nbr family     date                                                       
1         AUTOMOTIVE 2013-01-01        0            0     0.0     0.0     0.0   
                     2013-01-02     1782            0     1.0     0.0     0.0   
                     2013-01-03     3564            0     0.0     1.0     0.0   
                     2013-01-04     5346            0     0.0     0.0     1.0   
                     2013-01-05     7128            0     0.0     0.0     0.0   
...                                  ...          ...     ...     ...     ...   
9         SEAFOOD    2017-08-11  2993759            0     0.0     0.0     1.0   
                     2017-08-12  2995541            4     0.0     0.0     0.0   
                     2017-08-13  2997323            0     0.0     0.0     0.0   
                     2017-08-14  2999105            0     0.0     0.0     0.0   
                     2017-08-15  3000887            0     0.0     0.0     0.0   

                                 s(5,7)  s(6,7)  s(7,7)  sin(1,freq=YE-DEC)  \
store_nbr family     date                                                     
1         AUTOMOTIVE 2013-01-01     0.0     0.0     0.0            0.000000   
                     2013-01-02     0.0     0.0     0.0            0.017213   
                     2013-01-03     0.0     0.0     0.0            0.034422   
                     2013-01-04     0.0     0.0     0.0            0.051620   
                     2013-01-05     1.0     0.0     0.0            0.068802   
...                                 ...     ...     ...                 ...   
9         SEAFOOD    2017-08-11     0.0     0.0     0.0           -0.628763   
                     2017-08-12     1.0     0.0     0.0           -0.642055   
                     2017-08-13     0.0     1.0     0.0           -0.655156   
                     2017-08-14     0.0     0.0     1.0           -0.668064   
                     2017-08-15     0.0     0.0     0.0           -0.680773   

                                 cos(1,freq=YE-DEC)  sin(2,freq=YE-DEC)  \
store_nbr family     date                                                 
1         AUTOMOTIVE 2013-01-01            1.000000            0.000000   
                     2013-01-02            0.999852            0.034422   
                     2013-01-03            0.999407            0.068802   
                     2013-01-04            0.998667            0.103102   
                     2013-01-05            0.997630            0.137279   
...                                             ...                 ...   
9         SEAFOOD    2017-08-11           -0.777597            0.977848   
                     2017-08-12           -0.766659            0.984474   
                     2017-08-13           -0.755493            0.989932   
                     2017-08-14           -0.744104            0.994218  

In [12]:
# First we would create our Y
y = train_df.reset_index().set_index(["date", "id"]).sales
y

date        id     
2013-01-01  0           0.000000
2013-01-02  1782        2.000000
2013-01-03  3564        3.000000
2013-01-04  5346        3.000000
2013-01-05  7128        5.000000
                         ...    
2017-08-11  2993759    23.830999
2017-08-12  2995541    16.859001
2017-08-13  2997323    20.000000
2017-08-14  2999105    17.000000
2017-08-15  3000887    16.000000
Name: sales, Length: 3008016, dtype: float32

In [13]:
# test_set to make and combine with our dataset
test_pd = pd.read_csv("test.csv",
                      dtype = {'store_nbr' : 'category',
                               'family' : 'category',
                               'onpromotion' : 'uint32',
                              },
                      parse_dates = ["date"])

test_pd["date"] = test_pd.date.dt.to_period('D')

test_pd = test_pd.set_index(["date", "id"])
test_pd

store_nbr                      family  onpromotion
date       id                                                        
2017-08-16 3000888         1                  AUTOMOTIVE            0
           3000889         1                   BABY CARE            0
           3000890         1                      BEAUTY            2
           3000891         1                   BEVERAGES           20
           3000892         1                       BOOKS            0
...                      ...                         ...          ...
2017-08-31 3029395         9                     POULTRY            1
           3029396         9              PREPARED FOODS            0
           3029397         9                     PRODUCE            1
           3029398         9  SCHOOL AND OFFICE SUPPLIES            9
           3029399         9                     SEAFOOD            0

[28512 rows x 3 columns]

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
# Evaluate our models performance
def scoring(y, y_pred):
    mae = mean_absolute_error(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mse)
    return print(f"MAE: {mae:.5f}, RMSE: {rmse:.5f}, R^2:{r2:.5f}")

In [29]:
df_2017 = pd.DataFrame({
    'y' : y,
    'x_pred': xs_pred
})

df_2017 = df_2017.loc[pd.IndexSlice['2017', :], :]
df_2017

,,y,x_pred
date,id,,
2017-01-01,2596374,0.000000,209.198381
2017-01-02,2598156,5.000000,618.543230
2017-01-03,2599938,4.000000,467.521999
2017-01-04,2601720,1.000000,358.457216
2017-01-05,2603502,2.000000,378.828144
...,...,...,...
2017-08-11,2993759,23.830999,124.230338
2017-08-12,2995541,16.859001,399.399257
2017-08-13,2997323,20.000000,281.348821


In [16]:
# Extract numerical features from our date columns
treet = train_df.copy().reset_index()
treet["date"] = treet["date"].dt.to_timestamp()          # Convert Period to Timestamp

# For train set
treet["day"] = treet.date.dt.day
treet["month"] = treet.date.dt.month
treet["year"] = treet.date.dt.year
treet["dayofweek"] = treet.date.dt.dayofweek

# freq_encode family and store_nbr
treet["family"] = treet["family"].cat.codes
treet["store_nbr"] = treet["store_nbr"].cat.codes

treet = treet.set_index("id").drop('date', axis = 1)

In [17]:
# Extract numerical features from our date columns
tree_test = test_df.copy().reset_index()
tree_test["date"] = tree_test["date"].dt.to_timestamp()          # Convert Period to Timestamp

# For test set
tree_test["day"] = tree_test.date.dt.day
tree_test["month"] = tree_test.date.dt.month
tree_test["year"] = tree_test.date.dt.year
tree_test["dayofweek"] = tree_test.date.dt.dayofweek

# freq_encode family and store_nbr
tree_test["family"] = tree_test["family"].cat.codes
tree_test["store_nbr"] = tree_test["store_nbr"].cat.codes


tree_test = tree_test.set_index("id").drop('date', axis = 1)

In [18]:
# Use frequency encoding for the remaining categorical columns
cat_col = ['city', 'type', 'state', 'cluster']

for col in cat_col:
    freq = treet[col].value_counts(normalize = True)
    treet[col + '_freq'] = treet[col].map(freq)
    tree_test[col + '_freq'] = tree_test[col].map(freq)
    treet = treet.drop(col, axis = 1)
    tree_test = tree_test.drop(col, axis = 1)

tree_test

,store_nbr,family,const,trend,onpromotion,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)",dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3000888,0,0,1.0,1689.0,0,1.0,0.0,0.0,0.0,0.0,0.0,-0.693281,-0.720667,0.999250,0.038722,-0.746972,0.664855,0.077386,-0.997001,0.635432,0.772157,1.224647e-16,-1.000000,-2.449294e-16,1.000000,46.80,47.57,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.0,16,8,2017,2,0.333333,0.333333,0.351852,0.074074
3002670,0,0,1.0,1690.0,0,0.0,1.0,0.0,0.0,0.0,0.0,-0.705584,-0.708627,0.999991,0.004304,-0.711657,0.702527,0.008607,-0.999963,0.699458,0.714673,-6.824241e-02,-0.997669,1.361666e-01,0.990686,47.07,46.80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,17,8,2017,3,0.333333,0.333333,0.351852,0.074074
3004452,0,0,1.0,1691.0,0,0.0,0.0,1.0,0.0,0.0,0.0,-0.717677,-0.696376,0.999546,-0.030120,-0.674444,0.738326,-0.060213,-0.998186,0.758306,0.651899,-1.361666e-01,-0.990686,2.697968e-01,0.962917,48.59,47.07,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,18,8,2017,4,0.333333,0.333333,0.351852,0.074074
3006234,0,0,1.0,1692.0,0,0.0,0.0,0.0,1.0,0.0,0.0,-0.729558,-0.683919,0.997917,-0.064508,-0.635432,0.772157,-0.128748,-0.991677,0.811539,0.584298,-2.034560e-01,-0.979084,3.984011e-01,0.917211,48.19,48.59,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,19,8,2017,5,0.333333,0.333333,0.351852,0.074074
3008016,0,0,1.0,1693.0,0,0.0,0.0,0.0,0.0,1.0,0.0,-0.741222,-0.671260,0.995105,-0.098820,-0.594727,0.803928,-0.196673,-0.980469,0.858764,0.512371,-2.697968e-01,-0.962917,5.195840e-01,0.854419,47.79,48.19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,20,8,2017,6,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3022271,53,32,1.0,1700.0,0,0.0,0.0,0.0,0.0,1.0,0.0,-0.816538,-0.577292,0.942761,-0.333469,-0.271958,0.962309,-0.628763,-0.777597,0.997917,-0.064508,-6.825531e-01,-0.730836,9.976688e-01,0.068242,46.82,47.23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,27,8,2017,6,0.333333,0.148148,0.351852,0.111111
3024053,53,32,1.0,1701.0,0,0.0,0.0,0.0,0.0,0.0,1.0,-0.826354,-0.563151,0.930724,-0.365723,-0.221922,0.975065,-0.680773,-0.732494,0.988678,-0.150055,-7.308360e-01,-0.682553,9.976688e-01,-0.068242,46.40,46.82,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,28,8,2017,0,0.333333,0.148148,0.351852,0.111111
3025835,53,32,1.0,1702.0,0,0.0,0.0,0.0,0.0,0.0,0.0,-0.835925,-0.548843,0.917584,-0.397543,-0.171293,0.985220,-0.729558,-0.683919,0.972118,-0.234491,-7.757113e-01,-0.631088,9.790841e-01,-0.203456,46.46,46.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,29,8,2017,1,0.333333,0.148148,0.351852,0.111111


In [19]:
tree_x = treet.drop('sales', axis = 1)
tree_x

,store_nbr,family,const,trend,onpromotion,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)",dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0,0,1.0,1.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,93.14,94.88,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0.0,1,1,2013,1,0.333333,0.333333,0.351852,0.074074
1782,0,0,1.0,2.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667,0.068802,0.997630,0.085965,0.996298,0.069756,0.997564,0.139173,0.990268,93.14,93.14,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,2,1,2013,2,0.333333,0.333333,0.351852,0.074074
3564,0,0,1.0,3.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671,0.137279,0.990532,0.171293,0.985220,0.139173,0.990268,0.275637,0.961262,92.97,93.14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.0,3,1,2013,3,0.333333,0.333333,0.351852,0.074074
5346,0,0,1.0,4.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023,0.205104,0.978740,0.255353,0.966848,0.207912,0.978148,0.406737,0.913545,93.12,92.97,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,4,1,2013,4,0.333333,0.333333,0.351852,0.074074
7128,0,0,1.0,5.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740,0.271958,0.962309,0.337523,0.941317,0.275637,0.961262,0.529919,0.848048,93.15,93.12,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,5,1,2013,5,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993759,53,32,1.0,1684.0,0,0.0,0.0,1.0,0.0,0.0,0.0,-0.628763,-0.777597,0.977848,0.209315,-0.891981,0.452072,0.409356,-0.912375,0.255353,0.966848,0.334880,-0.942261,-0.631088,0.775711,48.81,48.54,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,11,8,2017,4,0.333333,0.148148,0.351852,0.111111
2995541,53,32,1.0,1685.0,4,0.0,0.0,0.0,1.0,0.0,0.0,-0.642055,-0.766659,0.984474,0.175531,-0.867456,0.497513,0.345612,-0.938377,0.337523,0.941317,0.269797,-0.962917,-0.519584,0.854419,48.40,48.81,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,12,8,2017,5,0.333333,0.148148,0.351852,0.111111
2997323,53,32,1.0,1686.0,0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655156,-0.755493,0.989932,0.141540,-0.840618,0.541628,0.280231,-0.959933,0.417194,0.908818,0.203456,-0.979084,-0.398401,0.917211,48.00,48.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,13,8,2017,6,0.333333,0.148148,0.351852,0.111111


In [20]:
tree_y = np.log1p(treet.sales)
tree_y

id
0          0.000000
1782       1.098612
3564       1.386294
5346       1.386294
7128       1.791759
             ...   
2993759    3.212093
2995541    2.882508
2997323    3.044523
2999105    2.890372
3000887    2.833213
Name: sales, Length: 3008016, dtype: float32

In [21]:
tree_x.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3008016 entries, 0 to 3000887
Data columns (total 67 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   store_nbr                int8    
 1   family                   int8    
 2   const                    float64 
 3   trend                    float64 
 4   onpromotion              uint32  
 5   s(2,7)                   float64 
 6   s(3,7)                   float64 
 7   s(4,7)                   float64 
 8   s(5,7)                   float64 
 9   s(6,7)                   float64 
 10  s(7,7)                   float64 
 11  sin(1,freq=YE-DEC)       float64 
 12  cos(1,freq=YE-DEC)       float64 
 13  sin(2,freq=YE-DEC)       float64 
 14  cos(2,freq=YE-DEC)       float64 
 15  sin(3,freq=YE-DEC)       float64 
 16  cos(3,freq=YE-DEC)       float64 
 17  sin(4,freq=YE-DEC)       float64 
 18  cos(4,freq=YE-DEC)       float64 
 19  sin(5,freq=YE-DEC)       float64 
 20  cos(5,freq=YE-DEC)       floa

In [22]:
tree_x.type_freq = tree_x.type_freq.astype(float)
tree_x

,store_nbr,family,const,trend,onpromotion,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)",dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0,0,1.0,1.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,93.14,94.88,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0.0,1,1,2013,1,0.333333,0.333333,0.351852,0.074074
1782,0,0,1.0,2.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667,0.068802,0.997630,0.085965,0.996298,0.069756,0.997564,0.139173,0.990268,93.14,93.14,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,2,1,2013,2,0.333333,0.333333,0.351852,0.074074
3564,0,0,1.0,3.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671,0.137279,0.990532,0.171293,0.985220,0.139173,0.990268,0.275637,0.961262,92.97,93.14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.0,3,1,2013,3,0.333333,0.333333,0.351852,0.074074
5346,0,0,1.0,4.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023,0.205104,0.978740,0.255353,0.966848,0.207912,0.978148,0.406737,0.913545,93.12,92.97,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,4,1,2013,4,0.333333,0.333333,0.351852,0.074074
7128,0,0,1.0,5.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740,0.271958,0.962309,0.337523,0.941317,0.275637,0.961262,0.529919,0.848048,93.15,93.12,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,5,1,2013,5,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993759,53,32,1.0,1684.0,0,0.0,0.0,1.0,0.0,0.0,0.0,-0.628763,-0.777597,0.977848,0.209315,-0.891981,0.452072,0.409356,-0.912375,0.255353,0.966848,0.334880,-0.942261,-0.631088,0.775711,48.81,48.54,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,11,8,2017,4,0.333333,0.148148,0.351852,0.111111
2995541,53,32,1.0,1685.0,4,0.0,0.0,0.0,1.0,0.0,0.0,-0.642055,-0.766659,0.984474,0.175531,-0.867456,0.497513,0.345612,-0.938377,0.337523,0.941317,0.269797,-0.962917,-0.519584,0.854419,48.40,48.81,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,12,8,2017,5,0.333333,0.148148,0.351852,0.111111
2997323,53,32,1.0,1686.0,0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655156,-0.755493,0.989932,0.141540,-0.840618,0.541628,0.280231,-0.959933,0.417194,0.908818,0.203456,-0.979084,-0.398401,0.917211,48.00,48.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,13,8,2017,6,0.333333,0.148148,0.351852,0.111111


In [23]:
tree_test.type_freq = tree_test.type_freq.astype(float)
tree_test

,store_nbr,family,const,trend,onpromotion,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)",dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3000888,0,0,1.0,1689.0,0,1.0,0.0,0.0,0.0,0.0,0.0,-0.693281,-0.720667,0.999250,0.038722,-0.746972,0.664855,0.077386,-0.997001,0.635432,0.772157,1.224647e-16,-1.000000,-2.449294e-16,1.000000,46.80,47.57,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.0,16,8,2017,2,0.333333,0.333333,0.351852,0.074074
3002670,0,0,1.0,1690.0,0,0.0,1.0,0.0,0.0,0.0,0.0,-0.705584,-0.708627,0.999991,0.004304,-0.711657,0.702527,0.008607,-0.999963,0.699458,0.714673,-6.824241e-02,-0.997669,1.361666e-01,0.990686,47.07,46.80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,17,8,2017,3,0.333333,0.333333,0.351852,0.074074
3004452,0,0,1.0,1691.0,0,0.0,0.0,1.0,0.0,0.0,0.0,-0.717677,-0.696376,0.999546,-0.030120,-0.674444,0.738326,-0.060213,-0.998186,0.758306,0.651899,-1.361666e-01,-0.990686,2.697968e-01,0.962917,48.59,47.07,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,18,8,2017,4,0.333333,0.333333,0.351852,0.074074
3006234,0,0,1.0,1692.0,0,0.0,0.0,0.0,1.0,0.0,0.0,-0.729558,-0.683919,0.997917,-0.064508,-0.635432,0.772157,-0.128748,-0.991677,0.811539,0.584298,-2.034560e-01,-0.979084,3.984011e-01,0.917211,48.19,48.59,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,19,8,2017,5,0.333333,0.333333,0.351852,0.074074
3008016,0,0,1.0,1693.0,0,0.0,0.0,0.0,0.0,1.0,0.0,-0.741222,-0.671260,0.995105,-0.098820,-0.594727,0.803928,-0.196673,-0.980469,0.858764,0.512371,-2.697968e-01,-0.962917,5.195840e-01,0.854419,47.79,48.19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,20,8,2017,6,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3022271,53,32,1.0,1700.0,0,0.0,0.0,0.0,0.0,1.0,0.0,-0.816538,-0.577292,0.942761,-0.333469,-0.271958,0.962309,-0.628763,-0.777597,0.997917,-0.064508,-6.825531e-01,-0.730836,9.976688e-01,0.068242,46.82,47.23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,27,8,2017,6,0.333333,0.148148,0.351852,0.111111
3024053,53,32,1.0,1701.0,0,0.0,0.0,0.0,0.0,0.0,1.0,-0.826354,-0.563151,0.930724,-0.365723,-0.221922,0.975065,-0.680773,-0.732494,0.988678,-0.150055,-7.308360e-01,-0.682553,9.976688e-01,-0.068242,46.40,46.82,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,28,8,2017,0,0.333333,0.148148,0.351852,0.111111
3025835,53,32,1.0,1702.0,0,0.0,0.0,0.0,0.0,0.0,0.0,-0.835925,-0.548843,0.917584,-0.397543,-0.171293,0.985220,-0.729558,-0.683919,0.972118,-0.234491,-7.757113e-01,-0.631088,9.790841e-01,-0.203456,46.46,46.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,29,8,2017,1,0.333333,0.148148,0.351852,0.111111


In [24]:
# Prepare the data for both models
lin_df = pd.read_csv("det_train.csv",
                     parse_dates = ['date']
                    )
lin_df['date'] = lin_df.date.dt.to_period('D')
lin_df = lin_df.set_index('date').drop('is_new_year', axis = 1)
lin_df

,const,trend,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)"
date,,,,,,,,,,,,,,,,,,,,,,
2013-01-01,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
2013-01-02,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667,0.068802,0.997630,0.085965,0.996298,0.069756,0.997564,0.139173,0.990268
2013-01-03,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671,0.137279,0.990532,0.171293,0.985220,0.139173,0.990268,0.275637,0.961262
2013-01-04,1.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023,0.205104,0.978740,0.255353,0.966848,0.207912,0.978148,0.406737,0.913545
2013-01-05,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740,0.271958,0.962309,0.337523,0.941317,0.275637,0.961262,0.529919,0.848048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-08-11,1.0,1684.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.628763,-0.777597,0.977848,0.209315,-0.891981,0.452072,0.409356,-0.912375,0.255353,0.966848,0.334880,-0.942261,-0.631088,0.775711
2017-08-12,1.0,1685.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.642055,-0.766659,0.984474,0.175531,-0.867456,0.497513,0.345612,-0.938377,0.337523,0.941317,0.269797,-0.962917,-0.519584,0.854419
2017-08-13,1.0,1686.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655156,-0.755493,0.989932,0.141540,-0.840618,0.541628,0.280231,-0.959933,0.417194,0.908818,0.203456,-0.979084,-0.398401,0.917211


In [25]:
lin_test = pd.read_csv('det_test.csv',
                       parse_dates = ['date']
                      )
lin_test['date'] = lin_test.date.dt.to_period('D')
lin_test = lin_test.set_index('date').drop('is_new_year', axis = 1)
lin_test

,const,trend,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)"
date,,,,,,,,,,,,,,,,,,,,,,
2017-08-16,1,1689,1,0,0,0,0,0,-0.693281,-0.720667,0.999250,0.038722,-0.746972,0.664855,0.077386,-0.997001,0.635432,0.772157,1.220000e-16,-1.000000,-2.450000e-16,1.000000
2017-08-17,1,1690,0,1,0,0,0,0,-0.705584,-0.708627,0.999991,0.004304,-0.711657,0.702527,0.008607,-0.999963,0.699458,0.714673,-6.824241e-02,-0.997669,1.361666e-01,0.990686
2017-08-18,1,1691,0,0,1,0,0,0,-0.717677,-0.696376,0.999546,-0.030120,-0.674444,0.738326,-0.060213,-0.998186,0.758306,0.651899,-1.361666e-01,-0.990686,2.697968e-01,0.962917
2017-08-19,1,1692,0,0,0,1,0,0,-0.729558,-0.683919,0.997917,-0.064508,-0.635432,0.772157,-0.128748,-0.991677,0.811539,0.584298,-2.034560e-01,-0.979084,3.984011e-01,0.917211
2017-08-20,1,1693,0,0,0,0,1,0,-0.741222,-0.671260,0.995105,-0.098820,-0.594727,0.803928,-0.196673,-0.980469,0.858764,0.512371,-2.697968e-01,-0.962917,5.195840e-01,0.854419
2017-08-21,1,1694,0,0,0,0,0,1,-0.752667,-0.658402,0.991114,-0.133015,-0.552435,0.833556,-0.263665,-0.964614,0.899631,0.436651,-3.348796e-01,-0.942261,6.310879e-01,0.775711
2017-08-22,1,1695,0,0,0,0,0,0,-0.763889,-0.645348,0.985948,-0.167052,-0.508671,0.860961,-0.329408,-0.944188,0.933837,0.357698,-3.984011e-01,-0.917211,7.308360e-01,0.682553
2017-08-23,1,1696,1,0,0,0,0,0,-0.774884,-0.632103,0.979614,-0.200891,-0.463550,0.886071,-0.393590,-0.919286,0.961130,0.276097,-4.600650e-01,-0.887885,8.169699e-01,0.576680
2017-08-24,1,1697,0,1,0,0,0,0,-0.785650,-0.618671,0.972118,-0.234491,-0.417194,0.908818,-0.455907,-0.890028,0.981306,0.192452,-5.195840e-01,-0.854419,8.878852e-01,0.460065


In [26]:
lin_y = train_df.sales.to_frame()
lin_y

sales
store_nbr family     date                 
1         AUTOMOTIVE 2013-01-01   0.000000
                     2013-01-02   2.000000
                     2013-01-03   3.000000
                     2013-01-04   3.000000
                     2013-01-05   5.000000
...                                    ...
9         SEAFOOD    2017-08-11  23.830999
                     2017-08-12  16.859001
                     2017-08-13  20.000000
                     2017-08-14  17.000000
                     2017-08-15  16.000000

[3008016 rows x 1 columns]

In [27]:
lin_y = lin_y.unstack(['store_nbr', 'family'])
lin_y

sales                                                \
store_nbr           1                                                 
family     AUTOMOTIVE BABY CARE BEAUTY BEVERAGES BOOKS BREAD/BAKERY   
date                                                                  
2013-01-01        0.0       0.0    0.0       0.0   0.0     0.000000   
2013-01-02        2.0       0.0    2.0    1091.0   0.0   470.652008   
2013-01-03        3.0       0.0    0.0     919.0   0.0   310.654999   
2013-01-04        3.0       0.0    3.0     953.0   0.0   198.365997   
2013-01-05        5.0       0.0    3.0    1160.0   0.0   301.057007   
...               ...       ...    ...       ...   ...          ...   
2017-08-11        1.0       0.0    1.0    1006.0   0.0   145.606995   
2017-08-12        6.0       0.0    3.0    1659.0   0.0   243.220001   
2017-08-13        1.0       0.0    1.0     803.0   0.0   136.679001   
2017-08-14        1.0       0.0    6.0    2201.0   0.0   346.037994   
2017-08-15        4.0       0.0    4.0    1942.0   0.0   329.541016   

                                                                        \
store_nbr                                                                
family     CELEBRATION CLEANING  DAIRY        DELI   EGGS FROZEN FOODS   
date                                                                     
2013-01-01         0.0      0.0    0.0    0.000000    0.0     0.000000   
2013-01-02         0.0   1060.0  579.0  164.069000  246.0   131.000000   
2013-01-03         0.0    836.0  453.0  151.582001  203.0    87.043999   
2013-01-04         0.0    827.0  460.0  131.410995  171.0    65.000000   
2013-01-05         0.0    811.0  464.0  118.612999  177.0    70.000000   
...                ...      ...    ...         ...    ...          ...   
2017-08-11         4.0    341.0  343.0   64.302002   86.0    61.000000   
2017-08-12         3.0    351.0  526.0   99.487999  113.0   107.793999   
2017-08-13         1.0    169.0  266.0   47.770000   60.0    50.000000   
2017-08-14         4.0    571.0  699.0  154.578003  170.0   110.000000   
2017-08-15        21.0    703.0  602.0  116.402000  131.0    89.000000   

                                                             \
store_nbr                                                     
family     GROCERY I GROCERY II HARDWARE HOME AND KITCHEN I   
date                                                          
2013-01-01       0.0        0.0      0.0                0.0   
2013-01-02    2652.0       31.0      3.0                0.0   
2013-01-03    2121.0       12.0      1.0                0.0   
2013-01-04    2056.0       15.0      7.0                0.0   
2013-01-05    2216.0       30.0      1.0                0.0   
...              ...        ...      ...                ...   
2017-08-11    1270.0        9.0      1.0               27.0   
2017-08-12    1630.0       19.0      0.0               17.0   
2017-08-13     952.0        6.0      1.0               13.0   
2017-08-14    2407.0       20.0      0.0               50.0   
2017-08-15    2508.0       13.0      3.0               30.0   

                                                                     \
store_nbr                                                             
family     HOME AND KITCHEN II HOME APPLIANCES HOME CARE LADIESWEAR   
date                                                                  
2013-01-01                 0.0             0.0       0.0        0.0   
2013-01-02                 0.0             0.0       0.0        0.0   
2013-01-03                 0.0             2.0       0.0        0.0   
2013-01-04                 0.0             0.0       0.0        0.0   
2013-01-05                 0.0             0.0       0.0        0.0   
...                        ...             ...       ...        ...   
2017-08-11                14.0             0.0      74.0        3.0   
2017-08-12                31.0             0.0     116.0        9.0   
2017-08-13                 8.0           

In [28]:
# Rmove all columns fed to Linear regression from XGboost dataset
lincol = lin_df.columns.to_list()
lincol

['const',
 'trend',
 's(2,7)',
 's(3,7)',
 's(4,7)',
 's(5,7)',
 's(6,7)',
 's(7,7)',
 'sin(1,freq=YE-DEC)',
 'cos(1,freq=YE-DEC)',
 'sin(2,freq=YE-DEC)',
 'cos(2,freq=YE-DEC)',
 'sin(3,freq=YE-DEC)',
 'cos(3,freq=YE-DEC)',
 'sin(4,freq=YE-DEC)',
 'cos(4,freq=YE-DEC)',
 'sin(5,freq=YE-DEC)',
 'cos(5,freq=YE-DEC)',
 'sin(1,freq=QE-DEC)',
 'cos(1,freq=QE-DEC)',
 'sin(2,freq=QE-DEC)',
 'cos(2,freq=QE-DEC)']

In [29]:
xg_df = tree_x.drop(lincol, axis = 1)
xg_df

,store_nbr,family,onpromotion,dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,93.14,94.88,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0.0,1,1,2013,1,0.333333,0.333333,0.351852,0.074074
1782,0,0,0,93.14,93.14,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,2,1,2013,2,0.333333,0.333333,0.351852,0.074074
3564,0,0,0,92.97,93.14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.0,3,1,2013,3,0.333333,0.333333,0.351852,0.074074
5346,0,0,0,93.12,92.97,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,4,1,2013,4,0.333333,0.333333,0.351852,0.074074
7128,0,0,0,93.15,93.12,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,5,1,2013,5,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993759,53,32,0,48.81,48.54,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,11,8,2017,4,0.333333,0.148148,0.351852,0.111111
2995541,53,32,4,48.40,48.81,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,12,8,2017,5,0.333333,0.148148,0.351852,0.111111
2997323,53,32,0,48.00,48.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,13,8,2017,6,0.333333,0.148148,0.351852,0.111111


In [30]:
tree_y

id
0          0.000000
1782       1.098612
3564       1.386294
5346       1.386294
7128       1.791759
             ...   
2993759    3.212093
2995541    2.882508
2997323    3.044523
2999105    2.890372
3000887    2.833213
Name: sales, Length: 3008016, dtype: float32

In [31]:
xg_test = tree_test.drop(lincol, axis = 1)
xg_test

,store_nbr,family,onpromotion,dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3000888,0,0,0,46.80,47.57,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.0,16,8,2017,2,0.333333,0.333333,0.351852,0.074074
3002670,0,0,0,47.07,46.80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,17,8,2017,3,0.333333,0.333333,0.351852,0.074074
3004452,0,0,0,48.59,47.07,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,18,8,2017,4,0.333333,0.333333,0.351852,0.074074
3006234,0,0,0,48.19,48.59,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,19,8,2017,5,0.333333,0.333333,0.351852,0.074074
3008016,0,0,0,47.79,48.19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,20,8,2017,6,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3022271,53,32,0,46.82,47.23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,27,8,2017,6,0.333333,0.148148,0.351852,0.111111
3024053,53,32,0,46.40,46.82,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,28,8,2017,0,0.333333,0.148148,0.351852,0.111111
3025835,53,32,0,46.46,46.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,29,8,2017,1,0.333333,0.148148,0.351852,0.111111


In [32]:
# Function to minus lin_y from lin_y_pred
def linear_model(lin_y, lin_y_pred, index, col, df):
    """
    This model helps create a residual even when data fed to the linear model and data fed to the tree model have different indexes.
    lin_y is the training target for the linear model
    lin_y_pred is the prediction of the linear model
    index is the index that would be used to create dataframe for the linear model's prediction
    col is the columns for the dataframe of the linear model's prediction
    df is the dataframe that would be used to reorder the dataframe of the residual of the linear model's predition 
    and the true y values so that there is no disparity in values when feeding with the xgboost model. 
    """
    lin_pred_df = pd.DataFrame(lin_y_pred,
                               index = index.index,
                               columns = col.columns)
    residuals = lin_y - lin_pred_df
    resid_long = residuals.stack(['store_nbr', 'family'])
    resid_long = resid_long.reorder_levels(['store_nbr', 'family', 'date']).sort_index()
    resid_long = resid_long.loc[df.index]
    # Change resid_long to have the same index as our tree model
    x_tr = df[['id']].merge(resid_long, left_index = True, right_index = True, how = 'outer')
    x_tr = x_tr.reset_index().set_index('id')
    return x_tr

## MODELLING

Like has been said above, a hybrid model will be created using LinearRegression and Xgboost Regressor

In [33]:
# Instantiate linear model
lin_model = LinearRegression(fit_intercept = False)
lin_model

LinearRegression(fit_intercept=False)

In [34]:
lin_model.fit(lin_df, lin_y)

LinearRegression(fit_intercept=False)

In [35]:
lin_y_pred = lin_model.predict(lin_df)
lin_y_pred

array([[ 2.30140421e+00,  0.00000000e+00,  1.74857998e+00, ...,
         4.12963723e+02, -8.94686899e+00,  1.25731839e+01],
       [ 1.97210686e+00,  0.00000000e+00,  1.75080234e+00, ...,
        -1.20311558e+02, -9.97716624e+00,  1.16483677e+01],
       [ 1.51468431e+00,  0.00000000e+00,  1.67036411e+00, ...,
        -1.39028699e+02, -1.08880426e+01,  1.50304287e+01],
       ...,
       [ 2.92155842e+00,  0.00000000e+00,  2.34916872e+00, ...,
         2.25080042e+03,  3.70511618e+01,  2.40213708e+01],
       [ 4.78476657e+00,  0.00000000e+00,  3.73203289e+00, ...,
         1.80784948e+03,  3.33277347e+01,  1.32461035e+01],
       [ 5.33200072e+00,  0.00000000e+00,  3.66311509e+00, ...,
         2.22640815e+03,  3.43943661e+01,  1.27031959e+01]])

In [36]:
lin_pred_df = pd.DataFrame(lin_y_pred,
                               index = lin_y.index,
                               columns = lin_y.columns)
lin_pred_df

sales                                                          \
store_nbr           1                                                           
family     AUTOMOTIVE BABY CARE    BEAUTY    BEVERAGES     BOOKS BREAD/BAKERY   
date                                                                            
2013-01-01   2.301404       0.0  1.748580   920.430339 -0.011133   258.306580   
2013-01-02   1.972107       0.0  1.750802  1116.828061  0.009581   323.890938   
2013-01-03   1.514684       0.0  1.670364   853.141031 -0.053724   268.913709   
2013-01-04   2.258371       0.0  1.473094  1014.524736 -0.021453   277.336012   
2013-01-05   2.186655       0.0  1.827303  1116.894107 -0.030535   267.734194   
...               ...       ...       ...          ...       ...          ...   
2017-08-11   5.369707       0.0  3.457267  2288.572536  0.282266   404.764055   
2017-08-12   5.257718       0.0  3.833044  2383.865624  0.272186   394.102063   
2017-08-13   2.921558       0.0  2.349169  1345.752389  0.204126   189.600211   
2017-08-14   4.784767       0.0  3.732033  2269.166257  0.273131   419.781412   
2017-08-15   5.332001       0.0  3.663115  2229.438425  0.284270   394.688550   

                                                                        \
store_nbr                                                                
family     CELEBRATION    CLEANING       DAIRY        DELI        EGGS   
date                                                                     
2013-01-01   -0.891600  673.741023  477.809260  108.767776  138.411833   
2013-01-02    0.271600  818.455719  605.566971  119.027755  167.946040   
2013-01-03    0.582094  663.297000  438.260156   97.486532  124.575495   
2013-01-04    2.589477  708.424952  496.278548  130.138392  155.644735   
2013-01-05   -2.121595  591.611479  530.850057  113.892779  168.708230   
...                ...         ...         ...         ...         ...   
2017-08-11   20.237534  701.858027  776.782630  153.744374  137.770407   
2017-08-12   15.163821  582.402116  811.813640  137.386901  150.135717   
2017-08-13    8.735227  224.616476  423.750259   62.277591   61.881377   
2017-08-14   19.695158  661.554870  781.100322  143.293824  136.184040   
2017-08-15   15.413126  677.754867  762.016841  134.027789  122.694178   

                                                                              \
store_nbr                                                                      
family     FROZEN FOODS    GROCERY I GROCERY II  HARDWARE HOME AND KITCHEN I   
date                                                                           
2013-01-01   281.172005  1965.614747  33.590141  1.451183           3.591806   
2013-01-02   267.305829  2305.991325  35.935794  1.506870           5.880218   
2013-01-03   230.949069  1757.980680  30.804804  1.422131           9.993582   
2013-01-04   252.895319  1934.105176  28.647664  1.383192           9.346404   
2013-01-05   225.618472  1846.979098  26.631060  1.589218           4.196146   
...                 ...          ...        ...       ...                ...   
2017-08-11   153.003099  2575.458989  19.939987  1.788454          32.534690   
2017-08-12   132.606043  2486.389672  18.247817  2.001582          27.098217   
2017-08-13    37.236306  1183.909270   5.249801  0.830145          16.512989   
2017-08-14    93.740264  2536.515389  17.709594  1.614419          24.157766   
2017-08-15   113.359328  2562.119806  22.158747  1.883694          26.267598   

                                                                       \
store_nbr                                                               
family     HOME AND KITCHEN II HOME APPLIANCES   HOME CARE LADIESWEAR   
date                                                                    
2013-01-01           -0.869734        0.145702   11.380805   3.940670   
2013-01-02           -1.061320        0.211251   22.910926   4.063523   
2013-01-03           -1.973632        0.238931    

In [37]:
lin_pred_df.stack(['store_nbr', 'family']).reorder_levels(['store_nbr', 'family', 'date']).sort_index()

C:\Users\USER\AppData\Local\Temp\ipykernel_880\389358559.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  lin_pred_df.stack(['store_nbr', 'family']).reorder_levels(['store_nbr', 'family', 'date']).sort_index()


sales
store_nbr family     date                 
1         AUTOMOTIVE 2013-01-01   2.301404
                     2013-01-02   1.972107
                     2013-01-03   1.514684
                     2013-01-04   2.258371
                     2013-01-05   2.186655
...                                    ...
9         SEAFOOD    2017-08-11  10.330447
                     2017-08-12  19.166984
                     2017-08-13  24.021371
                     2017-08-14  13.246103
                     2017-08-15  12.703196

[3008016 rows x 1 columns]

In [38]:
x_tr = linear_model(lin_y = lin_y, lin_y_pred = lin_y_pred, index = lin_y, col = lin_y, df = train_df)
x_tr

C:\Users\USER\AppData\Local\Temp\ipykernel_880\2583430939.py:16: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  resid_long = residuals.stack(['store_nbr', 'family'])


,store_nbr,family,date,sales
id,,,,
0,1,AUTOMOTIVE,2013-01-01,-2.301404
1782,1,AUTOMOTIVE,2013-01-02,0.027893
3564,1,AUTOMOTIVE,2013-01-03,1.485316
5346,1,AUTOMOTIVE,2013-01-04,0.741629
7128,1,AUTOMOTIVE,2013-01-05,2.813345
...,...,...,...,...
2993759,9,SEAFOOD,2017-08-11,13.500553
2995541,9,SEAFOOD,2017-08-12,-2.307983
2997323,9,SEAFOOD,2017-08-13,-4.021371


In [39]:
scoring(lin_y, lin_pred_df)

MAE: 87.37026, RMSE: 353.26798, R^2:0.38032


In [40]:
xg_y = x_tr.sales.to_frame()
xg_y

,sales
id,
0,-2.301404
1782,0.027893
3564,1.485316
5346,0.741629
7128,2.813345
...,...
2993759,13.500553
2995541,-2.307983
2997323,-4.021371


In [43]:
# Intantiate XGmodel and train
xg_mod = xg.XGBRegressor(
    random_state = 42,
    n_estimators = 1500,
    max_depth = 6,
    subsample= 0.8,
    colsample_bytree = 0.8,
    gamma = 0.3,
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    tree_method = 'hist',
    objective = 'reg:squarederror',
    early_stopping_rounds = 50,
    eval_metric = 'rmse',
    n_jobs = -1,
    verbosity = 1
)
xg_mod

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=50,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1500, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [44]:
# Train the Xgb model
split = 2918915
xg_mod.fit(xg_df.iloc[:split], xg_y.iloc[:split],
           eval_set = [(xg_df.iloc[split:], xg_y[split:])],
           verbose = True)

[0]	validation_0-rmse:370.93246
[1]	validation_0-rmse:364.52450
[2]	validation_0-rmse:358.28776
[3]	validation_0-rmse:355.34053
[4]	validation_0-rmse:349.34975
[5]	validation_0-rmse:346.89530
[6]	validation_0-rmse:344.59867
[7]	validation_0-rmse:342.62229
[8]	validation_0-rmse:339.84374
[9]	validation_0-rmse:339.07448
[10]	validation_0-rmse:337.15252
[11]	validation_0-rmse:336.56046
[12]	validation_0-rmse:336.26481
[13]	validation_0-rmse:331.80390
[14]	validation_0-rmse:325.31251
[15]	validation_0-rmse:319.35854
[16]	validation_0-rmse:318.24555
[17]	validation_0-rmse:317.42916
[18]	validation_0-rmse:316.89807
[19]	validation_0-rmse:316.59471
[20]	validation_0-rmse:316.30877
[21]	validation_0-rmse:316.02612
[22]	validation_0-rmse:315.84499
[23]	validation_0-rmse:314.89828
[24]	validation_0-rmse:313.01364
[25]	validation_0-rmse:312.83212
[26]	validation_0-rmse:311.02187
[27]	validation_0-rmse:309.19657
[28]	validation_0-rmse:309.10306
[29]	validation_0-rmse:309.25130
[30]	validation_0-rm

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=50,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1500, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [46]:
# Intantiate XGmodel and train
xg_model = xg.XGBRegressor(
    random_state = 42,
    n_estimators = 448,
    max_depth = 6,
    subsample= 0.8,
    colsample_bytree = 0.8,
    gamma = 0.3,
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    tree_method = 'hist',
    objective = 'reg:squarederror',
    n_jobs = -1,
    verbosity = 1
)
xg_model

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=448, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [47]:
xg_model.fit(xg_df, xg_y)

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=448, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [48]:
xg_pred = xg_model.predict(xg_df)
xg_pred

array([ 18.292913, -76.46892 , -23.67076 , ...,  22.426315, -16.459198,
       -26.131474], dtype=float32)

In [49]:
xgp_df = pd.DataFrame(xg_pred, index = xg_y.index, columns = ['XGB_pred'])

In [50]:
xgp_df

,XGB_pred
id,
0,18.292913
1782,-76.468918
3564,-23.670759
5346,-56.840210
7128,-38.687386
...,...
2993759,56.250439
2995541,-299.336609
2997323,22.426315


In [51]:
def wide_to_long(df, df2):
    long = df.stack(['store_nbr', 'family']) 
    long = long.reorder_levels(['store_nbr', 'family', 'date']).sort_index()
    long = long.loc[df2.index]

    # Change long to have the same index as our tree model
    tr = df2[['id']].merge(long, left_index = True, right_index = True, how = 'outer')
    tr = tr.reset_index().set_index('id')
    tr['LIN_pred'] = tr.sales
    linp_df = tr.LIN_pred.to_frame()
    return linp_df

In [52]:
linp_df = wide_to_long(lin_pred_df, train_df)
linp_df

C:\Users\USER\AppData\Local\Temp\ipykernel_880\2168376627.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long = df.stack(['store_nbr', 'family'])


,LIN_pred
id,
0,2.301404
1782,1.972107
3564,1.514684
5346,2.258371
7128,2.186655
...,...
2993759,10.330447
2995541,19.166984
2997323,24.021371


In [53]:
sub = pd.DataFrame()
sub['sales']= xgp_df['XGB_pred'] + linp_df['LIN_pred']
sub

,sales
id,
0,20.594318
1782,-74.496811
3564,-22.156075
5346,-54.581839
7128,-36.500730
...,...
2993759,66.580885
2995541,-280.169625
2997323,46.447686


In [54]:
scoring(tree_y, sub)

MAE: 373.12573, RMSE: 1129.17381, R^2:-175461.34375


In [55]:
# Create the prediction
lin_p = lin_model.predict(lin_test)
xg_p = xg_model.predict(xg_test)
# make xg_p a dataframe
xgr_sub = pd.DataFrame(xg_p, index = xg_test.index, columns = ['XGB_pred'])
xgr_sub

,XGB_pred
id,
3000888,20.847168
3002670,33.964916
3004452,23.592819
3006234,-31.327726
3008016,-110.200699
...,...
3022271,10.942481
3024053,-20.161854
3025835,-19.287447


In [56]:
lin_sub = wide_to_long(df = pd.DataFrame(lin_p,
                                    index = lin_test.index,
                                    columns = lin_y.columns), df2 = test_df)
lin_sub

C:\Users\USER\AppData\Local\Temp\ipykernel_880\2168376627.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long = df.stack(['store_nbr', 'family'])


,LIN_pred
id,
3000888,4.969832
3002670,4.478898
3004452,5.188625
3006234,5.082703
3008016,2.754747
...,...
3022271,25.166465
3024053,14.343206
3025835,13.747100


In [57]:
subm = pd.DataFrame()
subm['sales']= xgr_sub['XGB_pred'] + lin_sub['LIN_pred']
subm

,sales
id,
3000888,25.817000
3002670,38.443814
3004452,28.781445
3006234,-26.245023
3008016,-107.445952
...,...
3022271,36.108946
3024053,-5.818648
3025835,-5.540347


In [58]:
subm.to_csv('submislog.csv', index = True)

### LOG-TRANSFORMATION

Converting our targets into logged transformed values to improve the score of our XGB model in this hybrid cofiguration

> Using Logtransformed targets to trian our model for improvements

In [59]:
lin_df

,const,trend,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)"
date,,,,,,,,,,,,,,,,,,,,,,
2013-01-01,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
2013-01-02,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667,0.068802,0.997630,0.085965,0.996298,0.069756,0.997564,0.139173,0.990268
2013-01-03,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671,0.137279,0.990532,0.171293,0.985220,0.139173,0.990268,0.275637,0.961262
2013-01-04,1.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023,0.205104,0.978740,0.255353,0.966848,0.207912,0.978148,0.406737,0.913545
2013-01-05,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740,0.271958,0.962309,0.337523,0.941317,0.275637,0.961262,0.529919,0.848048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-08-11,1.0,1684.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.628763,-0.777597,0.977848,0.209315,-0.891981,0.452072,0.409356,-0.912375,0.255353,0.966848,0.334880,-0.942261,-0.631088,0.775711
2017-08-12,1.0,1685.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.642055,-0.766659,0.984474,0.175531,-0.867456,0.497513,0.345612,-0.938377,0.337523,0.941317,0.269797,-0.962917,-0.519584,0.854419
2017-08-13,1.0,1686.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655156,-0.755493,0.989932,0.141540,-0.840618,0.541628,0.280231,-0.959933,0.417194,0.908818,0.203456,-0.979084,-0.398401,0.917211


In [60]:
lin_test

,const,trend,"s(2,7)","s(3,7)","s(4,7)","s(5,7)","s(6,7)","s(7,7)","sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)","cos(4,freq=YE-DEC)","sin(5,freq=YE-DEC)","cos(5,freq=YE-DEC)","sin(1,freq=QE-DEC)","cos(1,freq=QE-DEC)","sin(2,freq=QE-DEC)","cos(2,freq=QE-DEC)"
date,,,,,,,,,,,,,,,,,,,,,,
2017-08-16,1,1689,1,0,0,0,0,0,-0.693281,-0.720667,0.999250,0.038722,-0.746972,0.664855,0.077386,-0.997001,0.635432,0.772157,1.220000e-16,-1.000000,-2.450000e-16,1.000000
2017-08-17,1,1690,0,1,0,0,0,0,-0.705584,-0.708627,0.999991,0.004304,-0.711657,0.702527,0.008607,-0.999963,0.699458,0.714673,-6.824241e-02,-0.997669,1.361666e-01,0.990686
2017-08-18,1,1691,0,0,1,0,0,0,-0.717677,-0.696376,0.999546,-0.030120,-0.674444,0.738326,-0.060213,-0.998186,0.758306,0.651899,-1.361666e-01,-0.990686,2.697968e-01,0.962917
2017-08-19,1,1692,0,0,0,1,0,0,-0.729558,-0.683919,0.997917,-0.064508,-0.635432,0.772157,-0.128748,-0.991677,0.811539,0.584298,-2.034560e-01,-0.979084,3.984011e-01,0.917211
2017-08-20,1,1693,0,0,0,0,1,0,-0.741222,-0.671260,0.995105,-0.098820,-0.594727,0.803928,-0.196673,-0.980469,0.858764,0.512371,-2.697968e-01,-0.962917,5.195840e-01,0.854419
2017-08-21,1,1694,0,0,0,0,0,1,-0.752667,-0.658402,0.991114,-0.133015,-0.552435,0.833556,-0.263665,-0.964614,0.899631,0.436651,-3.348796e-01,-0.942261,6.310879e-01,0.775711
2017-08-22,1,1695,0,0,0,0,0,0,-0.763889,-0.645348,0.985948,-0.167052,-0.508671,0.860961,-0.329408,-0.944188,0.933837,0.357698,-3.984011e-01,-0.917211,7.308360e-01,0.682553
2017-08-23,1,1696,1,0,0,0,0,0,-0.774884,-0.632103,0.979614,-0.200891,-0.463550,0.886071,-0.393590,-0.919286,0.961130,0.276097,-4.600650e-01,-0.887885,8.169699e-01,0.576680
2017-08-24,1,1697,0,1,0,0,0,0,-0.785650,-0.618671,0.972118,-0.234491,-0.417194,0.908818,-0.455907,-0.890028,0.981306,0.192452,-5.195840e-01,-0.854419,8.878852e-01,0.460065


In [61]:
lin_y

sales                                                \
store_nbr           1                                                 
family     AUTOMOTIVE BABY CARE BEAUTY BEVERAGES BOOKS BREAD/BAKERY   
date                                                                  
2013-01-01        0.0       0.0    0.0       0.0   0.0     0.000000   
2013-01-02        2.0       0.0    2.0    1091.0   0.0   470.652008   
2013-01-03        3.0       0.0    0.0     919.0   0.0   310.654999   
2013-01-04        3.0       0.0    3.0     953.0   0.0   198.365997   
2013-01-05        5.0       0.0    3.0    1160.0   0.0   301.057007   
...               ...       ...    ...       ...   ...          ...   
2017-08-11        1.0       0.0    1.0    1006.0   0.0   145.606995   
2017-08-12        6.0       0.0    3.0    1659.0   0.0   243.220001   
2017-08-13        1.0       0.0    1.0     803.0   0.0   136.679001   
2017-08-14        1.0       0.0    6.0    2201.0   0.0   346.037994   
2017-08-15        4.0       0.0    4.0    1942.0   0.0   329.541016   

                                                                        \
store_nbr                                                                
family     CELEBRATION CLEANING  DAIRY        DELI   EGGS FROZEN FOODS   
date                                                                     
2013-01-01         0.0      0.0    0.0    0.000000    0.0     0.000000   
2013-01-02         0.0   1060.0  579.0  164.069000  246.0   131.000000   
2013-01-03         0.0    836.0  453.0  151.582001  203.0    87.043999   
2013-01-04         0.0    827.0  460.0  131.410995  171.0    65.000000   
2013-01-05         0.0    811.0  464.0  118.612999  177.0    70.000000   
...                ...      ...    ...         ...    ...          ...   
2017-08-11         4.0    341.0  343.0   64.302002   86.0    61.000000   
2017-08-12         3.0    351.0  526.0   99.487999  113.0   107.793999   
2017-08-13         1.0    169.0  266.0   47.770000   60.0    50.000000   
2017-08-14         4.0    571.0  699.0  154.578003  170.0   110.000000   
2017-08-15        21.0    703.0  602.0  116.402000  131.0    89.000000   

                                                             \
store_nbr                                                     
family     GROCERY I GROCERY II HARDWARE HOME AND KITCHEN I   
date                                                          
2013-01-01       0.0        0.0      0.0                0.0   
2013-01-02    2652.0       31.0      3.0                0.0   
2013-01-03    2121.0       12.0      1.0                0.0   
2013-01-04    2056.0       15.0      7.0                0.0   
2013-01-05    2216.0       30.0      1.0                0.0   
...              ...        ...      ...                ...   
2017-08-11    1270.0        9.0      1.0               27.0   
2017-08-12    1630.0       19.0      0.0               17.0   
2017-08-13     952.0        6.0      1.0               13.0   
2017-08-14    2407.0       20.0      0.0               50.0   
2017-08-15    2508.0       13.0      3.0               30.0   

                                                                     \
store_nbr                                                             
family     HOME AND KITCHEN II HOME APPLIANCES HOME CARE LADIESWEAR   
date                                                                  
2013-01-01                 0.0             0.0       0.0        0.0   
2013-01-02                 0.0             0.0       0.0        0.0   
2013-01-03                 0.0             2.0       0.0        0.0   
2013-01-04                 0.0             0.0       0.0        0.0   
2013-01-05                 0.0             0.0       0.0        0.0   
...                        ...             ...       ...        ...   
2017-08-11                14.0             0.0      74.0        3.0   
2017-08-12                31.0             0.0     116.0        9.0   
2017-08-13                 8.0           

In [62]:
xg_df

,store_nbr,family,onpromotion,dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,93.14,94.88,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0.0,1,1,2013,1,0.333333,0.333333,0.351852,0.074074
1782,0,0,0,93.14,93.14,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,2,1,2013,2,0.333333,0.333333,0.351852,0.074074
3564,0,0,0,92.97,93.14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.0,3,1,2013,3,0.333333,0.333333,0.351852,0.074074
5346,0,0,0,93.12,92.97,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,4,1,2013,4,0.333333,0.333333,0.351852,0.074074
7128,0,0,0,93.15,93.12,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,5,1,2013,5,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2993759,53,32,0,48.81,48.54,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,11,8,2017,4,0.333333,0.148148,0.351852,0.111111
2995541,53,32,4,48.40,48.81,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,12,8,2017,5,0.333333,0.148148,0.351852,0.111111
2997323,53,32,0,48.00,48.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,13,8,2017,6,0.333333,0.148148,0.351852,0.111111


In [63]:
xg_test

,store_nbr,family,onpromotion,dcoilwtico,dcoilwtico_lag,is_national_holiday,is_regional_holiday,is_local_holiday,holiday_type_Additional,holiday_type_Bridge,holiday_type_Event,holiday_type_Holiday,holiday_type_Transfer,holiday_type_Work Day,holiday_transferred,is_holiday_lead,is_holiday,is_holiday_lag,is_christmas_lead(2),is_christmas_lead(1),is_christmas,is_christmas_lag(1),is_christmas_lag(2),is_christmas_lag(3),is_new_year_lead(3),is_new_year_lead(2),is_new_year_lead(1),is_new_year,is_new_year_lag(1),is_new_year_lag(2),is_payday_15,is_payday_end,is_payday_lead,is_payday,is_payday_lag,earthquake_impact,earthquake_decay,day,month,year,dayofweek,city_freq,type_freq,state_freq,cluster_freq
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3000888,0,0,0,46.80,47.57,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.0,16,8,2017,2,0.333333,0.333333,0.351852,0.074074
3002670,0,0,0,47.07,46.80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,17,8,2017,3,0.333333,0.333333,0.351852,0.074074
3004452,0,0,0,48.59,47.07,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,18,8,2017,4,0.333333,0.333333,0.351852,0.074074
3006234,0,0,0,48.19,48.59,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,19,8,2017,5,0.333333,0.333333,0.351852,0.074074
3008016,0,0,0,47.79,48.19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,20,8,2017,6,0.333333,0.333333,0.351852,0.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3022271,53,32,0,46.82,47.23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,27,8,2017,6,0.333333,0.148148,0.351852,0.111111
3024053,53,32,0,46.40,46.82,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,28,8,2017,0,0.333333,0.148148,0.351852,0.111111
3025835,53,32,0,46.46,46.40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,29,8,2017,1,0.333333,0.148148,0.351852,0.111111


In [64]:
lin_logy = np.log1p(lin_y)
lin_logy

sales                                                   \
store_nbr           1                                                    
family     AUTOMOTIVE BABY CARE    BEAUTY BEVERAGES BOOKS BREAD/BAKERY   
date                                                                     
2013-01-01   0.000000       0.0  0.000000  0.000000   0.0     0.000000   
2013-01-02   1.098612       0.0  1.098612  6.995766   0.0     6.156241   
2013-01-03   1.386294       0.0  0.000000  6.824374   0.0     5.741897   
2013-01-04   1.386294       0.0  1.386294  6.860664   0.0     5.295142   
2013-01-05   1.791759       0.0  1.386294  7.057037   0.0     5.710616   
...               ...       ...       ...       ...   ...          ...   
2017-08-11   0.693147       0.0  0.693147  6.914731   0.0     4.987755   
2017-08-12   1.945910       0.0  1.386294  7.414573   0.0     5.498069   
2017-08-13   0.693147       0.0  0.693147  6.689599   0.0     4.924925   
2017-08-14   0.693147       0.0  1.945910  7.697121   0.0     5.849434   
2017-08-15   1.609438       0.0  1.609438  7.571989   0.0     5.800731   

                                                                             \
store_nbr                                                                     
family     CELEBRATION  CLEANING     DAIRY      DELI      EGGS FROZEN FOODS   
date                                                                          
2013-01-01    0.000000  0.000000  0.000000  0.000000  0.000000     0.000000   
2013-01-02    0.000000  6.966967  6.363028  5.106364  5.509388     4.882802   
2013-01-03    0.000000  6.729824  6.118097  5.027702  5.318120     4.477837   
2013-01-04    0.000000  6.719013  6.133398  4.885911  5.147494     4.189655   
2013-01-05    0.000000  6.699501  6.142037  4.784262  5.181784     4.262680   
...                ...       ...       ...       ...       ...          ...   
2017-08-11    1.609438  5.834811  5.840641  4.179023  4.465908     4.127134   
2017-08-12    1.386294  5.863631  6.267200  4.610038  4.736198     4.689456   
2017-08-13    0.693147  5.135798  5.587249  3.887115  4.110874     3.931826   
2017-08-14    1.609438  6.349139  6.551080  5.047147  5.141664     4.709530   
2017-08-15    3.091043  6.556778  6.401917  4.765604  4.882802     4.499810   

                                                              \
store_nbr                                                      
family     GROCERY I GROCERY II  HARDWARE HOME AND KITCHEN I   
date                                                           
2013-01-01  0.000000   0.000000  0.000000           0.000000   
2013-01-02  7.883446   3.465736  1.386294           0.000000   
2013-01-03  7.660114   2.564949  0.693147           0.000000   
2013-01-04  7.629004   2.772589  2.079442           0.000000   
2013-01-05  7.703910   3.433987  0.693147           0.000000   
...              ...        ...       ...                ...   
2017-08-11  7.147559   2.302585  0.693147           3.332205   
2017-08-12  7.396949   2.995732  0.000000           2.890372   
2017-08-13  6.859615   1.945910  0.693147           2.639057   
2017-08-14  7.786552   3.044523  0.000000           3.931826   
2017-08-15  7.827640   2.639057  1.386294           3.433987   

                                                                     \
store_nbr                                                             
family     HOME AND KITCHEN II HOME APPLIANCES HOME CARE LADIESWEAR   
date                                                                  
2013-01-01            0.000000        0.000000  0.000000   0.000000   
2013-01-02            0.000000        0.000000  0.000000   0.000000   
2013-01-03            0.000000        1.098612  0.000000   0.000000   
2013-01-04            0.000000        0.000000  0.000000   0.000000   
2013-01-05            0.000000        0.000000  0.000000   0.000000   
...                        ...             ...       ...        ...   
2017-08-11            2.708050        0.000000  

In [65]:
hy_model1 = LinearRegression(fit_intercept = False)
hy_model1 

LinearRegression(fit_intercept=False)

In [66]:
hy_model1.fit(lin_df, lin_logy)

LinearRegression(fit_intercept=False)

In [67]:
lin_pred2 = hy_model1.predict(lin_df)
lin_pred2

array([[ 0.95840347,  0.        ,  0.85253809, ...,  0.16981633,
        -0.43012353,  2.36712762],
       [ 0.88461497,  0.        ,  0.84891017, ..., -0.14450666,
        -0.47924095,  2.28592312],
       [ 0.79187101,  0.        ,  0.82978261, ..., -0.17937007,
        -0.57996993,  2.46960819],
       ...,
       [ 1.08188696,  0.        ,  0.99912476, ...,  8.65488071,
         2.81992479,  3.02625072],
       [ 1.65139111,  0.        ,  1.45761445, ...,  8.50188052,
         2.58302764,  2.50432089],
       [ 1.76246104,  0.        ,  1.4634371 , ...,  8.73363466,
         2.58586898,  2.49026729]])

In [68]:
lin_log = pd.DataFrame(lin_pred2, 
                       index = lin_y.index,
                       columns = lin_y.columns)
lin_log

sales                                                       \
store_nbr           1                                                        
family     AUTOMOTIVE BABY CARE    BEAUTY BEVERAGES     BOOKS BREAD/BAKERY   
date                                                                         
2013-01-01   0.958403       0.0  0.852538  6.332169 -0.032880     5.197308   
2013-01-02   0.884615       0.0  0.848910  6.465697 -0.021505     5.388785   
2013-01-03   0.791871       0.0  0.829783  6.330986 -0.050694     5.262294   
2013-01-04   0.941098       0.0  0.741516  6.423942 -0.030269     5.275723   
2013-01-05   0.979852       0.0  0.861188  6.576870 -0.042080     5.316707   
...               ...       ...       ...       ...       ...          ...   
2017-08-11   1.745989       0.0  1.390432  7.668896  0.160767     5.894940   
2017-08-12   1.771963       0.0  1.516423  7.800540  0.146853     5.920342   
2017-08-13   1.081887       0.0  0.999125  6.883784  0.121466     5.028355   
2017-08-14   1.651391       0.0  1.457614  7.716865  0.151030     5.983006   
2017-08-15   1.762461       0.0  1.463437  7.654016  0.163336     5.891104   

                                                                             \
store_nbr                                                                     
family     CELEBRATION  CLEANING     DAIRY      DELI      EGGS FROZEN FOODS   
date                                                                          
2013-01-01    0.037920  6.077504  5.728300  4.383820  4.626408     4.785803   
2013-01-02    0.122829  6.286517  5.925602  4.481361  4.829275     4.937762   
2013-01-03    0.154678  6.097024  5.695997  4.322196  4.540770     4.740935   
2013-01-04    0.223620  6.154661  5.782771  4.554327  4.763218     4.937539   
2013-01-05   -0.015531  6.060233  5.909753  4.504994  4.901438     4.905150   
...                ...       ...       ...       ...       ...          ...   
2017-08-11    3.199416  6.427640  6.565479  4.938626  4.835129     4.888810   
2017-08-12    2.944433  6.315134  6.679416  4.878128  4.958058     4.870089   
2017-08-13    2.199782  5.374651  5.823084  4.011781  4.173007     3.873727   
2017-08-14    2.909494  6.420876  6.629609  4.910669  4.863006     4.538583   
2017-08-15    3.001436  6.418808  6.571481  4.824047  4.752762     4.568078   

                                                              \
store_nbr                                                      
family     GROCERY I GROCERY II  HARDWARE HOME AND KITCHEN I   
date                                                           
2013-01-01  7.036838   3.245397  0.671416           0.358947   
2013-01-02  7.198011   3.333269  0.690784           0.516419   
2013-01-03  6.991874   3.170780  0.667065           0.554556   
2013-01-04  7.063388   3.023399  0.653237           0.598415   
2013-01-05  7.129399   3.032880  0.718607           0.485437   
...              ...        ...       ...                ...   
2017-08-11  7.720821   2.826691  0.847408           3.469452   
2017-08-12  7.768584   2.839524  0.915538           3.326540   
2017-08-13  6.891444   1.879455  0.449211           2.506456   
2017-08-14  7.774895   2.806201  0.787029           3.153690   
2017-08-15  7.745358   2.977987  0.872632           3.283763   

                                                                     \
store_nbr                                                             
family     HOME AND KITCHEN II HOME APPLIANCES HOME CARE LADIESWEAR   
date                                                                  
2013-01-01            0.045154        0.091147  0.168288   0.229931   
2013-01-02            0.033187        0.132378  0.255903   0.251746   
2013-01-03            0.020587        0.140981  0.108780   0.168992   
2013-01-04           -0.011290        0.118259  0.087653   0.103260   
2013-01-05            0.016627        0.127903  0.194085   0.227651   
...                        ...             ...       ...   

In [69]:
scoring(lin_logy, lin_log)

MAE: 0.50530, RMSE: 0.83202, R^2:0.37599


In [70]:
xg_logy = linear_model(lin_y = lin_logy, lin_y_pred = lin_log, index = lin_logy, col = lin_logy, df = train_df)
xg_logy

C:\Users\USER\AppData\Local\Temp\ipykernel_880\2583430939.py:16: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  resid_long = residuals.stack(['store_nbr', 'family'])


,store_nbr,family,date,sales
id,,,,
0,1,AUTOMOTIVE,2013-01-01,-0.958403
1782,1,AUTOMOTIVE,2013-01-02,0.213997
3564,1,AUTOMOTIVE,2013-01-03,0.594423
5346,1,AUTOMOTIVE,2013-01-04,0.445196
7128,1,AUTOMOTIVE,2013-01-05,0.811907
...,...,...,...,...
2993759,9,SEAFOOD,2017-08-11,0.927561
2995541,9,SEAFOOD,2017-08-12,0.020906
2997323,9,SEAFOOD,2017-08-13,0.018272


In [71]:
xgb_logy = xg_logy.sales.to_frame()
xgb_logy

,sales
id,
0,-0.958403
1782,0.213997
3564,0.594423
5346,0.445196
7128,0.811907
...,...
2993759,0.927561
2995541,0.020906
2997323,0.018272


In [82]:
# Instantiate XGBoost
xg_mod2 = xg.XGBRegressor(
    random_state = 42,
    n_estimator = 1500,
    max_depth = 6,
    subsample = 1.0,
    colsample_bytree = 0.8,
    gamma = 0.3, 
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    tree_method = 'hist',
    objective = 'reg:squarederror',
    eval_metric = 'rmse',
    early_stopping_rounds = 70,
    n_jobs = -1,
    verbosity = 1
)

xg_mod2

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=70,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimator=1500, n_estimators=None,
             n_jobs=-1, num_parallel_tree=None, ...)

In [83]:
# Train the Xgb model on log transformed target
cut = 2918915

xg_mod2.fit(xg_df.iloc[:cut], xgb_logy.iloc[:cut],
           eval_set = [(xg_df.iloc[cut:], xgb_logy[cut:])],
           verbose = True)

C:\Users\USER\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [03:07:26] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_estimator" } are not used.

  warnings.warn(smsg, UserWarning)


[0]	validation_0-rmse:0.83217
[1]	validation_0-rmse:0.80851
[2]	validation_0-rmse:0.79980
[3]	validation_0-rmse:0.78830
[4]	validation_0-rmse:0.78074
[5]	validation_0-rmse:0.77229
[6]	validation_0-rmse:0.76933
[7]	validation_0-rmse:0.76568
[8]	validation_0-rmse:0.75571
[9]	validation_0-rmse:0.74832
[10]	validation_0-rmse:0.74592
[11]	validation_0-rmse:0.74381
[12]	validation_0-rmse:0.73739
[13]	validation_0-rmse:0.73543
[14]	validation_0-rmse:0.71506
[15]	validation_0-rmse:0.70056
[16]	validation_0-rmse:0.69359
[17]	validation_0-rmse:0.69050
[18]	validation_0-rmse:0.68921
[19]	validation_0-rmse:0.68886
[20]	validation_0-rmse:0.68347
[21]	validation_0-rmse:0.68293
[22]	validation_0-rmse:0.68236
[23]	validation_0-rmse:0.69761
[24]	validation_0-rmse:0.69701
[25]	validation_0-rmse:0.69650
[26]	validation_0-rmse:0.69432
[27]	validation_0-rmse:0.69293
[28]	validation_0-rmse:0.69023
[29]	validation_0-rmse:0.68998
[30]	validation_0-rmse:0.68929
[31]	validation_0-rmse:0.68324
[32]	validation_0-

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=70,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimator=1500, n_estimators=None,
             n_jobs=-1, num_parallel_tree=None, ...)

In [84]:
print("Best Iteration: ", xg_mod2.best_iteration)
print("Best Score: ", xg_mod2.best_score)

Best Iteration:  98
Best Score:  0.6039170293379023


In [85]:
# Instantiate XGBoost
xg_mol = xg.XGBRegressor(
    n_estimator = 98,
    learning_rate = 0.05,
    max_depth = 6,
    subsample = 1.0,
    colsample_bytree = 0.8,
    gamma = 0.3, 
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    tree_method = 'hist',
    objective = 'reg:squarederror',
    random_state = 42,
    n_jobs = -1,
    verbosity = 0
)
xg_mol

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimator=98, n_estimators=None, n_jobs=-1,
             num_parallel_tree=None, ...)

In [87]:
xg_mol.fit(xg_df, xgb_logy)

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.3, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimator=98, n_estimators=None, n_jobs=-1,
             num_parallel_tree=None, ...)

In [88]:
xg_pred2 = xg_mol.predict(xg_df)
xg_pred2

array([-1.2728244 ,  0.25569427,  0.16095081, ..., -0.081709  ,
       -0.10242006, -0.10331468], dtype=float32)

In [89]:
xgpr_df = pd.DataFrame(xg_pred2, index = xgb_logy.index, columns = ['XGB_pred'])
xgpr_df

,XGB_pred
id,
0,-1.272824
1782,0.255694
3564,0.160951
5346,0.130306
7128,0.092491
...,...
2993759,-0.017697
2995541,0.026954
2997323,-0.081709


In [90]:
linpr_df = wide_to_long(lin_log, train_df)
linpr_df

C:\Users\USER\AppData\Local\Temp\ipykernel_880\2168376627.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long = df.stack(['store_nbr', 'family'])


,LIN_pred
id,
0,0.958403
1782,0.884615
3564,0.791871
5346,0.941098
7128,0.979852
...,...
2993759,2.284532
2995541,2.861601
2997323,3.026251


In [91]:
subms = pd.DataFrame()
subms['sales']= xgpr_df['XGB_pred'] + linpr_df['LIN_pred']
subms

,sales
id,
0,-0.314421
1782,1.140309
3564,0.952822
5346,1.071404
7128,1.072343
...,...
2993759,2.266834
2995541,2.888555
2997323,2.944542


In [93]:
scoring(tree_y, subms)

MAE: 0.46169, RMSE: 0.71020, R^2:0.93059


In [94]:
# Create the prediction
lin_pr = hy_model1.predict(lin_test)
xg_pr = xg_mol.predict(xg_test)
# make xg_p a dataframe
xgr_subm = pd.DataFrame(xg_pr, index = xg_test.index, columns = ['XGB_pred'])
xgr_subm

,XGB_pred
id,
3000888,-0.098235
3002670,-0.106855
3004452,-0.106855
3006234,-0.096995
3008016,-0.080268
...,...
3022271,-0.121731
3024053,-0.151568
3025835,-0.151568


In [95]:
lin_subm = wide_to_long(df = pd.DataFrame(lin_p,
                                    index = lin_test.index,
                                    columns = lin_y.columns), df2 = test_df)
lin_subm

C:\Users\USER\AppData\Local\Temp\ipykernel_880\2168376627.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long = df.stack(['store_nbr', 'family'])


,LIN_pred
id,
3000888,4.969832
3002670,4.478898
3004452,5.188625
3006234,5.082703
3008016,2.754747
...,...
3022271,25.166465
3024053,14.343206
3025835,13.747100


In [96]:
submn = pd.DataFrame()
submn['sales']= xgr_subm['XGB_pred'] + lin_subm['LIN_pred']
submn

,sales
id,
3000888,4.871597
3002670,4.372043
3004452,5.081770
3006234,4.985708
3008016,2.674479
...,...
3022271,25.044735
3024053,14.191637
3025835,13.595532


In [97]:
submn.to_csv('logsubmis.csv', index = True)